# IPE as a classifier of the IOI ground-truth nodes

Recreates the two appendix figures of *IPE: Isolating Path Effects for Improving Latent Circuit
Identification* (EMNLP'25 workshop):

| Figure | File | Setting |
|---|---|---|
| `fig:performances`    | `images/performance_as_classifier.png`    | IPE with **path ablation**, metric = target logit difference (%) |
| `fig:performances_CF` | `images/performance_as_classifier_CF.png` | IPE with **path counterfactual**, metric = indirect effect |

Both plot precision / recall / accuracy / F1 of the discovered node set against the
Wang et al. (2023) IOI circuit, as a function of the minimum path-contribution threshold, with
dashed verticals marking the percentage of nodes retained w.r.t. the full (lowest-threshold) circuit.

### Provenance of the data

The per-path scores behind the published figures were produced in the shared-task repo
(`nepp1d0/MIB-circuit-track-with-paths`) and are **not checked into this repository** — only the
aggregated MIB submissions are (`experiments/MIB/results/.../scores.json`, which carry edge scores
and node membership but no per-path score, so the threshold sweep cannot be re-derived from them).

This notebook therefore **re-runs the search** with the configuration recorded in
`experiments/MIB/results/*/*/gpt2*/results.json`, caches the scored paths to disk, and does the
sweep from there. If you still have the original `.pkl` of `(score, path)` tuples, drop it in
`PATH_CACHE_DIR` under the file name printed by `paths_cache_file(cfg)` and the search step is
skipped.

**Cost.** The counterfactual run (`min_contribution = 1e-4`, `indirect_effect`) took ~6 100 s on an
A100. The ablation run is heavier — the logged runs at the *coarser* threshold 0.1 were cut off by a
25 h cap. Set `QUICK = True` to exercise the whole pipeline in minutes at a coarser threshold.


In [ ]:
import os
import sys
import json
import time
import pickle as pkl
from dataclasses import dataclass, field, asdict

import numpy as np
import matplotlib.pyplot as plt

EXPERIMENTS_DIR = os.getcwd()                      # this notebook lives in experiments/
REPO_ROOT = os.path.dirname(EXPERIMENTS_DIR)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)                  # only needed if `ipe` is not pip-installed

PATH_CACHE_DIR = os.path.join(EXPERIMENTS_DIR, "detected_paths")
os.makedirs(PATH_CACHE_DIR, exist_ok=True)

# QUICK = True runs a much coarser (and much cheaper) search so the pipeline can be exercised
# end to end. The published figures need QUICK = False.
QUICK = False


## 1. Experiment configurations

Taken verbatim from the logged runs, except for `min_contribution`, which is set to the left edge of
each published x-axis (`0.025` for ablation, `1e-4` for counterfactual — the logged ablation run used
the coarser `0.1`).

`include_negative=False` implements *"selecting only paths that positively contribute to the correct
token's logit"* (appendix, §Relevance of the Retrieved Paths).


In [ ]:
@dataclass
class SearchConfig:
    name: str
    title: str
    metric: str                 # ipe metric name
    min_contribution: float
    cf: bool                    # counterfactual patching (vs zero ablation)
    denoising: bool = True
    include_negative: bool = False
    algorithm: str = "PathMessagePatching"
    search_strategy: str = "Threshold"
    model: str = "gpt2-small"
    task: str = "ioi"
    target_length: int = 15
    batch_size: int = 8
    positional: bool = False
    batch_heads: bool = True
    # thresholds (in units of the metric) at which the published x-axis ends
    x_max: float = None

ABLATION = SearchConfig(
    name="ablation",
    title="IPE with path ablation",
    metric="target_logit_percentage",   # = "Target Logit Difference (%)" of Eq. (6)
    min_contribution=0.025,
    cf=False,
    denoising=False,
    x_max=10.0,
)

COUNTERFACTUAL = SearchConfig(
    name="counterfactual",
    title="IPE with path counterfactual",
    metric="indirect_effect",
    min_contribution=1e-4,
    cf=True,
    denoising=True,
    x_max=0.035,
)

if QUICK:
    # ~2 orders of magnitude fewer paths; the curves keep their shape but stop much earlier on the left
    ABLATION.min_contribution, ABLATION.batch_size = 2.5, 4
    COUNTERFACTUAL.min_contribution, COUNTERFACTUAL.batch_size = 1e-2, 4

def paths_cache_file(cfg: SearchConfig) -> str:
    stem = (f"paths_{cfg.model}_{cfg.task}_{cfg.algorithm}_{cfg.search_strategy}_"
            f"{cfg.metric}_cf{cfg.cf}_pos{cfg.positional}_th{cfg.min_contribution:g}"
            f"_bs{cfg.batch_size}_len{cfg.target_length}.pkl")
    return os.path.join(PATH_CACHE_DIR, stem)

print(paths_cache_file(ABLATION))
print(paths_cache_file(COUNTERFACTUAL))


## 2. Ground truth

The manually reverse-engineered IOI circuit of Wang et al. (2023) — 26 attention heads in 7 classes.
The appendix excludes the Negative Name Movers, so the ground-truth set has 24 heads.

(The same dict lives in `experiments/tree_vs_path.py`; it is repeated here so the notebook is
self-contained.)


In [ ]:
IOI_CIRCUIT = {
    "Name Mover": [(9, 9), (10, 0), (9, 6)],
    "Negative Name Mover": [(10, 7), (11, 10)],
    "Backup Name Mover": [(10, 10), (10, 6), (10, 2), (10, 1), (11, 2), (9, 7), (9, 0), (11, 9)],
    "S-Inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
    "Induction": [(5, 5), (5, 8), (5, 9), (6, 9)],
    "Duplicate Token": [(0, 1), (0, 10), (3, 0)],
    "Previous Token": [(2, 2), (4, 11)],
}

EXCLUDE_NEGATIVE_NAME_MOVERS = True   # as stated in the appendix

def ground_truth_nodes(exclude_negative: bool = EXCLUDE_NEGATIVE_NAME_MOVERS) -> set[str]:
    """Ground-truth node labels, in the `a{layer}.h{head}` naming used by the MIB submissions."""
    gt = set()
    for group, heads in IOI_CIRCUIT.items():
        if exclude_negative and group == "Negative Name Mover":
            continue
        gt |= {f"a{l}.h{h}" for l, h in heads}
    return gt

GT = ground_truth_nodes()
print(len(GT), "ground-truth nodes:", sorted(GT))


## 3. Running the search (cached)

Mirrors `experiments/MIB/run_search.py`, including the clean/counterfactual inversion used for IOI
(clean and counterfactual prompts solve the same task with different names, so the roles can be
swapped and the counterfactual run becomes the base run).

Skipped entirely when the cache file already exists.


In [ ]:
def load_ioi_batch(model, cfg: SearchConfig):
    """Prompts / targets / counterfactuals of exactly `cfg.target_length` tokens, as in run_search.py."""
    from datasets import load_dataset

    prompts, answers, cf_prompts, cf_answers = [], [], [], []
    ds = load_dataset("mib-bench/ioi", split="train")
    cf_key = "s2_io_flip_counterfactual"
    for sample in ds:
        if model.to_tokens(sample["prompt"], prepend_bos=True).shape[1] != cfg.target_length:
            continue
        prompts.append(sample["prompt"])
        cf_prompts.append(sample[cf_key]["prompt"])
        answers.append(f' {sample["metadata"]["indirect_object"]}')
        cf_answers.append(f' {sample[cf_key]["choices"][sample[cf_key]["answerKey"]]}')
        if len(prompts) >= cfg.batch_size:
            break
    return prompts, answers, cf_prompts, cf_answers


def run_search(cfg: SearchConfig, force: bool = False) -> str:
    """Run IPE with `cfg` and cache the scored paths. Returns the cache path."""
    out = paths_cache_file(cfg)
    if os.path.exists(out) and not force:
        print(f"cache hit: {out}")
        return out

    import torch
    from transformer_lens import HookedTransformer
    from ipe.experiment import ExperimentManager

    device = torch.device("cuda" if torch.cuda.is_available() else
                          "mps" if torch.backends.mps.is_available() else "cpu")
    model = HookedTransformer.from_pretrained(cfg.model, device=device,
                                              torch_dtype=torch.float32, center_unembed=True)
    model.eval()

    prompts, answers, cf_prompts, cf_answers = load_ioi_batch(model, cfg)

    algorithm_params = {
        "min_contribution": cfg.min_contribution,
        "include_negative": cfg.include_negative,
        "batch_heads": cfg.batch_heads,
    }

    experiment = ExperimentManager(
        model=model,
        # IOI-specific inversion, see run_search.py
        prompts=cf_prompts if cfg.cf else prompts,
        targets=cf_answers if cfg.cf else answers,
        cf_prompts=prompts if cfg.cf else [],
        cf_targets=answers if cfg.cf else [],
        algorithm=cfg.algorithm,
        search_strategy=cfg.search_strategy,
        algorithm_params=algorithm_params,
        metric=cfg.metric,
        metric_params={},
        positional_search=cfg.positional,
        patch_type="counterfactual" if cfg.cf else "zero",
        patch_clean_into_cf=cfg.denoising,
    )

    t0 = time.perf_counter()
    experiment.run()
    elapsed = time.perf_counter() - t0
    experiment.save_paths(filepath=out)          # clean=True: drops caches/gradients
    with open(out.replace(".pkl", ".meta.json"), "w") as f:
        json.dump({**asdict(cfg), "paths_found": len(experiment.paths),
                   "time_seconds": elapsed, "device": str(device)}, f, indent=4)
    print(f"{len(experiment.paths)} paths in {elapsed:.0f}s -> {out}")
    return out


## 4. From paths to per-node scores

A node enters the circuit at threshold `t` iff it lies on at least one retained path, i.e. iff its
**best path score** is `>= t`. Recording that maximum per node turns the whole threshold sweep into a
comparison on a length-|nodes| vector.

Node naming follows `experiments/MIB/create_submission.py::path_to_nodes`; `input` and `logits` are
dropped since they are on every path by construction and are not part of the ground truth.


In [ ]:
def path_to_node_labels(path) -> set[str]:
    """`(score, path)` element -> set of MIB-style node labels. Mirrors create_submission.path_to_nodes."""
    from ipe.nodes import EMBED_Node, ATTN_Node, MLP_Node, FINAL_Node

    labels = set()
    for node in path:
        if isinstance(node, EMBED_Node):
            labels.add("input")
        elif isinstance(node, ATTN_Node):
            labels.add(f"a{node.layer}.h{node.head}")
        elif isinstance(node, MLP_Node):
            labels.add(f"m{node.layer}")
        elif isinstance(node, FINAL_Node):
            labels.add("logits")
        else:
            raise ValueError(f"Unknown node type: {type(node)}")
    return labels


BOUNDARY_NODES = {"input", "logits"}

def node_best_scores(paths_file: str) -> dict[str, float]:
    """label -> max |contribution| over the paths that contain it."""
    with open(paths_file, "rb") as f:
        paths = pkl.load(f)
    best: dict[str, float] = {}
    for score, path in paths:
        s = abs(float(score))
        for label in path_to_node_labels(path) - BOUNDARY_NODES:
            if s > best.get(label, -np.inf):
                best[label] = s
    print(f"{len(paths)} paths -> {len(best)} nodes")
    return best


## 5. Threshold sweep

At threshold `t` the predicted circuit is `C(t) = {n : best_score[n] >= t}`.

Precision, recall and F1 only involve `C(t)` and the ground truth. **Accuracy also needs a negative
class**, i.e. an explicit universe of candidate nodes, and the paper does not state which one it used.
Two defensible choices are provided:

* `"discovered"` (default) — the pool the search actually ranked: `C(t_min) ∪ GT`. Accuracy then
  answers *"of the nodes IPE considered relevant at some threshold, how many are classified
  correctly?"*, and the flat right-hand tail of the curve sits at `1 - |GT| / |universe|`.
* `"model"` — every attention head and MLP of the model (156 for GPT-2 small). Much more forgiving,
  since most heads are never proposed by the search.

The curves for precision, recall and F1 are identical under both; only accuracy moves.


In [ ]:
N_LAYERS, N_HEADS = 12, 12   # gpt2-small

def model_universe(n_layers: int = N_LAYERS, n_heads: int = N_HEADS) -> set[str]:
    heads = {f"a{l}.h{h}" for l in range(n_layers) for h in range(n_heads)}
    return heads | {f"m{l}" for l in range(n_layers)}


def sweep(best: dict[str, float], gt: set[str], universe: str = "discovered") -> dict[str, np.ndarray]:
    """Precision/recall/F1/accuracy and retained-node fraction over the full threshold range."""
    labels = np.array(sorted(best))
    scores = np.array([best[l] for l in labels])
    is_gt = np.array([l in gt for l in labels])

    if universe == "discovered":
        n_universe = len(set(labels) | gt)
    elif universe == "model":
        n_universe = len(model_universe() | gt)
    else:
        raise ValueError(universe)

    n_gt = len(gt)
    n_full = len(labels)  # circuit at the lowest threshold

    # every distinct score is a step of the curve; add a point past the last one (empty circuit)
    thresholds = np.unique(scores)
    thresholds = np.append(thresholds, thresholds[-1] * 1.0001)

    tp = np.array([int(np.sum(is_gt & (scores >= t))) for t in thresholds])
    predicted = np.array([int(np.sum(scores >= t)) for t in thresholds])
    fp = predicted - tp
    fn = n_gt - tp
    tn = n_universe - tp - fp - fn

    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.where(predicted > 0, tp / np.maximum(predicted, 1), 0.0)
        recall = tp / n_gt
        f1 = np.where(tp > 0, 2 * tp / np.maximum(predicted + n_gt, 1), 0.0)
    accuracy = (tp + tn) / n_universe

    return {
        "threshold": thresholds,
        "precision": 100 * precision,
        "recall": 100 * recall,
        "f1": 100 * f1,
        "accuracy": 100 * accuracy,
        "retained": 100 * predicted / n_full,
        "predicted": predicted,
        "tp": tp,
        "n_universe": n_universe,
        "n_gt": n_gt,
        "n_full": n_full,
    }


def retained_crossings(res: dict, levels=(50, 25, 20, 15, 10)) -> list[tuple[float, int]]:
    """First threshold at which the retained-node percentage drops to (at most) each level."""
    out = []
    for lv in levels:
        idx = np.argmax(res["retained"] <= lv)
        if res["retained"][idx] <= lv:
            out.append((float(res["threshold"][idx]), lv))
    return out


## 6. Plot


In [ ]:
def plot_classifier_curves(res: dict, title: str, x_max: float = None,
                           levels=(50, 25, 20, 15, 10), ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5.5))

    # repeat the last (empty-circuit) row at x_max so the flat right-hand tail is drawn
    x = res["threshold"]
    tail = x_max is not None and x_max > x[-1]
    if tail:
        x = np.append(x, x_max)
    def curve(key):
        y = res[key]
        return np.append(y, y[-1]) if tail else y

    ax.plot(x, curve("precision"), color="tab:green", lw=1.0, label="Precision", drawstyle="steps-post")
    ax.plot(x, curve("recall"), color="tab:blue", lw=1.0, alpha=0.6, label="Recall", drawstyle="steps-post")
    ax.plot(x, curve("accuracy"), color="gray", lw=2.5, label="Accuracy", drawstyle="steps-post")
    ax.plot(x, curve("f1"), color="red", lw=2.5, label="F1 Score", drawstyle="steps-post")

    best_acc = int(np.argmax(res["accuracy"]))
    best_f1 = int(np.argmax(res["f1"]))
    ax.plot(res["threshold"][best_acc], res["accuracy"][best_acc], marker="*", ms=18, mfc="white", mec="black",
            ls="none", label=(f"Best Accuracy: {res['accuracy'][best_acc]:.1f}% "
                              f"({res['retained'][best_acc]:.1f}% retained nodes)"))
    ax.plot(res["threshold"][best_f1], res["f1"][best_f1], marker="*", ms=18, mfc="orange", mec="red",
            ls="none", label=(f"Best F1 Score: {res['f1'][best_f1]:.1f}% "
                              f"({res['retained'][best_f1]:.1f}% retained nodes)"))

    for th, lv in retained_crossings(res, levels):
        ax.axvline(th, color="gray", ls="--", lw=0.8, alpha=0.7)
        ax.text(th, 2, f"{lv}%", color="gray", fontsize=9, ha="center", va="bottom",
                bbox=dict(boxstyle="square,pad=0.05", fc="white", ec="none"))
    ax.plot([], [], color="gray", ls="--", lw=0.8,
            label="Percentage of retained nodes from our circuit")

    ax.set_xscale("log")
    ax.set_xlim(x[0], x_max if x_max else x[-1])
    ax.set_ylim(0, 100)
    ax.set_xlabel("Minimum Contribution Threshold")
    ax.set_ylabel("Performance (%)")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False, fontsize=9)
    return ax


def summarise(res: dict, name: str):
    b_acc, b_f1 = int(np.argmax(res["accuracy"])), int(np.argmax(res["f1"]))
    print(f"[{name}] universe={res['n_universe']} nodes | ground truth={res['n_gt']} | "
          f"full circuit={res['n_full']} nodes")
    print(f"  best accuracy {res['accuracy'][b_acc]:.1f}% at threshold {res['threshold'][b_acc]:.5g} "
          f"({res['retained'][b_acc]:.1f}% retained, {res['predicted'][b_acc]} nodes, "
          f"recall {res['recall'][b_acc]:.1f}%, precision {res['precision'][b_acc]:.1f}%)")
    print(f"  best F1       {res['f1'][b_f1]:.1f}% at threshold {res['threshold'][b_f1]:.5g} "
          f"({res['retained'][b_f1]:.1f}% retained, {res['predicted'][b_f1]} nodes, "
          f"recall {res['recall'][b_f1]:.1f}%, precision {res['precision'][b_f1]:.1f}%)")
    print(f"  recall at the lowest threshold: {res['recall'][0]:.1f}% "
          f"({res['tp'][0]}/{res['n_gt']} ground-truth nodes)")


## 7. Figure 1 — path ablation (`performance_as_classifier.png`)


In [ ]:
ABLATION_PATHS = run_search(ABLATION)
abl_best = node_best_scores(ABLATION_PATHS)
abl = sweep(abl_best, GT, universe="discovered")
summarise(abl, "ablation")
plot_classifier_curves(abl, "IPE with path ablation as a classifier of the IOI ground-truth nodes",
                       x_max=ABLATION.x_max)
plt.show()


## 8. Figure 2 — path counterfactual (`performance_as_classifier_CF.png`)


In [ ]:
CF_PATHS = run_search(COUNTERFACTUAL)
cf_best = node_best_scores(CF_PATHS)
cf = sweep(cf_best, GT, universe="discovered")
summarise(cf, "counterfactual")
plot_classifier_curves(cf, "IPE with path counterfactual as a classifier of the IOI ground-truth nodes",
                       x_max=COUNTERFACTUAL.x_max)
plt.show()


## 9. Cross-checks against the paper

* *"the union of the circuits obtained using the two methods yields perfect recall"* (appendix, last
  paragraph) — checked below.
* *"Our circuits retained 47.6% of the original ground-truth edges, while capturing 97.5% of the
  ground-truth nodes"* (§Results, at an indirect-effect threshold of 1e-4) — node recall of the
  counterfactual run at its lowest threshold.


In [ ]:
union_recall = 100 * len((set(abl_best) | set(cf_best)) & GT) / len(GT)
print(f"recall of the union of the two circuits: {union_recall:.1f}%")
missed = GT - (set(abl_best) | set(cf_best))
print("missed by both:", sorted(missed) if missed else "none")
print(f"ablation-only ground-truth nodes:       {sorted((set(abl_best) & GT) - set(cf_best))}")
print(f"counterfactual-only ground-truth nodes: {sorted((set(cf_best) & GT) - set(abl_best))}")


### On the accuracy denominator

Precision, recall and F1 are fully determined by the search output and the ground truth. Accuracy is
not: it depends on the universe of candidate nodes, which the paper does not spell out and which
cannot be recovered from the artefacts in this repository. The published tails
(≈76% for ablation, ≈74.7% for counterfactual, both reached once the circuit is empty and accuracy
collapses to `1 - |GT| / |universe|`) imply a universe of roughly 100–120 nodes, i.e. the
*discovered* pool rather than all 156 components of GPT-2 small — hence the default above. Rerun
with `universe="model"` to see the alternative.


In [ ]:
for name, best in (("ablation", abl_best), ("counterfactual", cf_best)):
    for uni in ("discovered", "model"):
        r = sweep(best, GT, universe=uni)
        print(f"{name:15s} universe={uni:11s} (n={r['n_universe']:3d})  "
              f"best acc {r['accuracy'].max():5.1f}%   empty-circuit acc {r['accuracy'][-1]:5.1f}%   "
              f"best F1 {r['f1'].max():5.1f}%")
